In [153]:
import pandas as pd
import requests
from datetime import datetime
import numpy as np
import plotly.graph_objects as go

from messari_wrapper import fetch_user_engagement, fetch_all_users, fetch_user_mindshare
from messari_wrapper import get_user_by_id

## Get all the users

Exploration function. Used to get all the users and experiment with the ID of the Messari API

In [155]:
df_users = fetch_all_users(max_users=2000) 

Fetched page 1, total users so far: 2000


# Compare shiller mindshare overtime

In [29]:
benchmark_users = ['@mrpunkdoteth','@JakeGagain', '@villainmonkey', '@stoolpresidente']
always_sharing_tickers_users = ['@HopiumPapi', '@SrPetersETH', '@JakeGagain', '@cryptobeastreal', '@rovercrc', '@hasbulla_main']

top_ticker = ['@hasbulla_main', '@JMilei', '@stoolpresidente', '@mrpunkdoteth']
libra_insiders = ['@gianinaskarlett', '@SolJakey', '@notthreadguy', '@frankdegods', '@Banks']


## Engagement overtime

The 4 users I used as a benchmark to describe the token launch peak.

It includes also the normalization function to see the peak corresponding to the token launch

In [157]:
def normalize_engagement(df):
    """Normalize the engagement column to scale 0-1"""
    df['normalized_engagement'] = (df['engagement'] - df['engagement'].min()) / (df['engagement'].max() - df['engagement'].min())
    return df

In [158]:
top_ticker = ['@hasbulla_main', '@JMilei', '@stoolpresidente', '@mrpunkdoteth']


In [159]:
fig = go.Figure()

# Custom color palette
colors = ['#FF6B6B',    # Bright Coral
          '#00FFC6',    # Bright Turquoise
          '#BB86FC',    # Bright Purple
          '#FFDE03'] 

for i, user in enumerate(always_sharing_tickers_users):
    print(f"Fetching engagement data for {user}...")
    try:
        df_engagement = fetch_user_engagement(user)
        
        fig.add_trace(
            go.Scatter(
                x=df_engagement['timestamp'],
                y=df_engagement['engagement'],
                mode='lines+markers',
                name=user,
                line=dict(width=3, color=colors[i % len(colors)]),
                marker=dict(size=6, symbol='circle')
            )
        )

    except requests.exceptions.HTTPError as e:
        print(f"Error fetching data for {user}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

fig.update_layout(
    title={
        'text': 'User Engagement Trends',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24, color='white')
    },
    xaxis_title='Date',
    yaxis_title='Normalized Engagement',
    legend_title='Users',
    template='plotly_dark',
    plot_bgcolor='rgba(0,0,0,0)',    # Transparent background
    paper_bgcolor='rgba(0,0,0,0)',   # Transparent paper
    font=dict(
        family="Helvetica",
        color='white'
    ),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',     # Transparent legend
        bordercolor='rgba(255,255,255,0.3)',
        borderwidth=1,
        x=1.02,
        y=1,
        orientation='v'
    ),
    margin=dict(l=80, r=80, t=100, b=80)
)

fig.update_xaxes(
    tickformat='%Y-%m-%d',
    dtick='M1',
    ticklabelmode='period',
    gridcolor='rgba(255,255,255,0.1)',
    gridwidth=0.5,
    showline=True,
    linewidth=1,
    linecolor='rgba(255,255,255,0.3)'
)

fig.update_yaxes(
    title_standoff=20,
    tickformat='.2f',
    gridcolor='rgba(255,255,255,0.1)',
    gridwidth=0.5,
    showline=True,
    linewidth=1,
    linecolor='rgba(255,255,255,0.3)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(255,255,255,0.2)'
)

fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Engagement: %{y:.3f}<extra></extra>'
)

fig.show()

Fetching engagement data for @HopiumPapi...
Fetching engagement data for @SrPetersETH...
Fetching engagement data for @JakeGagain...
Fetching engagement data for @rovercrc...


## Before and After stats

This section will help you if you wanna know how shilling a coin affected the performances of a specific account.

• Set the accounts in the libra_insider list

• Set the split date

• Select the days before and after

In [163]:
def calculate_medians(fig_data, split_date):
    all_y_before = []
    all_y_after = []
    
    for trace in fig_data:
        x_data = trace.x
        y_data = trace.y
        
        # Split data into before and after
        before_mask = x_data <= split_date
        after_mask = x_data > split_date
        
        all_y_before.extend(y_data[before_mask])
        all_y_after.extend(y_data[after_mask])
    
    return np.median(all_y_before), np.median(all_y_after)

In [164]:
libra_insiders = ['@gianinaskarlett', '@SolJakey']

split_date = datetime(2025, 2, 14)  # Libra launch date
days_before_after = 40

In [165]:
fig = go.Figure()


colors = ['#FF6B6B',    # Bright Coral
          '#00FFC6',    # Bright Turquoise
          '#BB86FC',    # Bright Purple
          '#FFDE03'] 

for i, user in enumerate(libra_insiders):
    print(f"Fetching engagement data for {user}...")
    try:
        df_engagement = fetch_user_engagement(user)
        # Get only before and after the split date
        #df_engagement['timestamp'] = pd.to_datetime(df_engagement['timestamp'])
        df_engagement = df_engagement[(df_engagement['timestamp'] >= split_date - pd.Timedelta(days=days_before_after)) &
                                      (df_engagement['timestamp'] <= split_date + pd.Timedelta(days=days_before_after))]
        #df_engagement = normalize_engagement(df_engagement)
        
        fig.add_trace(
            go.Scatter(
                x=df_engagement['timestamp'],
                y=df_engagement['engagement'],
                mode='lines+markers',
                name=user,
                line=dict(width=3, color=colors[i % len(colors)]),
                marker=dict(size=6, symbol='circle')
            )
        )
        engagement_before, engagement_after = calculate_medians(fig.data, split_date)
        print(f"User: {user}")
        print(f"Median engagement before {split_date}: {engagement_before}")
        print(f"Median engagement after {split_date}: {engagement_after}")

    except requests.exceptions.HTTPError as e:
        print(f"Error fetching data for {user}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

fig.update_layout(

    xaxis_title='Date',
    yaxis_title='Normalized Engagement',
    legend_title='Users',
    template='plotly_dark',
    plot_bgcolor='rgba(0,0,0,0)',    # Transparent background
    paper_bgcolor='rgba(0,0,0,0)',   # Transparent paper
    font=dict(
        family="Helvetica",
        color='white'
    ),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',     # Transparent legend
        bordercolor='rgba(255,255,255,0.3)',
        borderwidth=1,
        x=1.02,
        y=1,
        orientation='v'
    ),
    margin=dict(l=80, r=80, t=100, b=80)
)

fig.update_xaxes(
    tickformat='%Y-%m-%d',
    dtick='M1',
    ticklabelmode='period',
    gridcolor='rgba(255,255,255,0.1)',
    gridwidth=0.5,
    showline=True,
    linewidth=1,
    linecolor='rgba(255,255,255,0.3)'
)

fig.update_yaxes(
    title_standoff=20,
    tickformat='.2f',
    gridcolor='rgba(255,255,255,0.1)',
    gridwidth=0.5,
    showline=True,
    linewidth=1,
    linecolor='rgba(255,255,255,0.3)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(255,255,255,0.2)'
)

fig.update_layout(
    # ...existing layout configuration...
    shapes=[
        # Vertical line at split date
        dict(
            type='line',
            x0=split_date,
            x1=split_date,
            y0=0,
            y1=1,
            yref='paper',
            line=dict(
                color='rgba(255, 255, 255, 0.5)',
                width=2,
                dash='dash'
            )
        ),
        # Rectangle for before split date
        dict(
            type='rect',
            x0=fig.data[0].x.min(),  # Start of data
            x1=split_date,
            y0=0,
            y1=1,
            yref='paper',
            fillcolor='rgba(255, 0, 0, 0.2)',
            line_width=0,
            layer='below'
        ),
        # Rectangle for after split date
        dict(
            type='rect',
            x0=split_date,
            x1=fig.data[0].x.max(),  # End of data
            y0=0,
            y1=1,
            yref='paper',
            fillcolor='rgba(0, 255, 0, 0.2)',
            line_width=0,
            layer='below'
        )
    ],
    
)

# Add annotation for the split date
fig.add_annotation(
    x=split_date,
    y=1.05,
    yref='paper',
    text='$LIBRA Launch',
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor='rgba(255, 255, 255, 0.5)'
)


fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Engagement: %{y:.3f}<extra></extra>'
)

fig.show()

Fetching engagement data for @gianinaskarlett...
User: @gianinaskarlett
Median engagement before 2025-02-14 00:00:00: 120.092652712963
Median engagement after 2025-02-14 00:00:00: 110.6654090498085
Fetching engagement data for @SolJakey...
User: @SolJakey
Median engagement before 2025-02-14 00:00:00: 99.6338440354775
Median engagement after 2025-02-14 00:00:00: 89.119995952655


## Mindshare chart

Mindshare overtime for accounts

In [166]:
constant_shillers = ['@HopiumPapi', '@JakeGagain', '@cryptobeastreal', '@shahh']

#detectives = ['@CryptoRugMunch']

In [167]:
# Create figure
fig = go.Figure()

# Custom neon color palette for better visibility on dark background
colors = [
    '#00ff00', '#ff00ff', '#00ffff', '#ffff00', '#ff3366',
    '#66ff33', '#ff6600', '#33ccff', '#ff99cc', '#99ff99'
]

# Add traces for each username
for idx, username in enumerate(constant_shillers):
    print(f"Fetching mindshare data for {username}...")
    try:
        mindshare = fetch_user_mindshare(username)
    except requests.exceptions.HTTPError as e:
        print(f"Error fetching data for {username}: {e}")
        continue
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        continue

    fig.add_trace(
        go.Scatter(
            x=mindshare['timestamp'],
            y=mindshare['rank'],
            name=username,
            mode='lines',
            line=dict(
                width=2.5,
                color=colors[idx % len(colors)],
            ),
            hovertemplate='Rank: %{y}<br>Date: %{x}<extra></extra>'
        )
    )

# Update layout with transparent background
fig.update_layout(
    title={
        'text': 'Mindshare Rankings Over Time',
        'font': {'size': 24, 'color': 'white'},
        'y': 0.95
    },
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    yaxis_title='Rank',
    xaxis_title='Date',
    yaxis_autorange='reversed',
    height=700,
    width=1200,
    showlegend=True,
    legend=dict(
        font=dict(color='white', size=12),
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(255,255,255,0.2)',
        borderwidth=1,
        title=dict(text='Username', font=dict(color='white', size=14))
    ),
    hovermode='x unified',
    hoverlabel=dict(bgcolor='rgba(0,0,0,0.8)', font_size=12),
    xaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(255,255,255,0.1)',
        tickfont=dict(color='white'),
        title_font=dict(color='white', size=14),
        showline=False,
        linewidth=2,
        linecolor='rgba(255,255,255,0.2)'
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(255,255,255,0.1)',
        tickfont=dict(color='white'),
        title_font=dict(color='white', size=14),
        showline=False,  # Remove the top line
        linewidth=2,
        linecolor='rgba(255,255,255,0.2)'
    )
)
# Add a subtle margin around the plot
fig.update_layout(margin=dict(l=60, r=30, t=80, b=60))

# Show the plot
fig.show()

Fetching mindshare data for @HopiumPapi...
Fetching mindshare data for @JakeGagain...
Fetching mindshare data for @cryptobeastreal...
Fetching mindshare data for @shahh...


## Index of growth

The goal of this section is to create an index for all those shiller account where I have an "average" mindshare, compared with my average mindshare.

In [168]:
def get_index_data(user_list):

    index_data = []
    for idx, username in enumerate(user_list):
        print(f"Fetching mindshare data for {username}...")
        try:
            mindshare = fetch_user_mindshare(username)
            index_data.append(mindshare)
        except requests.exceptions.HTTPError as e:
            print(f"Error fetching data for {username}: {e}")
            continue
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            continue
    
    return index_data

def create_index(index_data):

    start_date = '2025-03-01' # I chose this date because the data is available for every user

    df_index = pd.concat(index_data)
    df_index = df_index[df_index['timestamp'] >= start_date]
    df_index['timestamp'] = pd.to_datetime(df_index['timestamp'])
    df_index.set_index(df_index['timestamp'], inplace=True)
    df_index = df_index[['mindshare']]
    df_index['index_mindshare'] = df_index.resample('1D').mean()
    df_index.sort_index(inplace=True)

    return df_index

In [ ]:
constant_shillers = ['@HopiumPapi', '@JakeGagain', '@cryptobeastreal',
                     '@elchartox', '@issathecooker', '@realdogen', '@ProTheDoge',
                    '@shahh', '@TeTheGamer', '@moneyl0rd', '@MarcellxMarcell', '@OfficialTravlad']

index_data = get_index_data(constant_shillers)
df_shillers = create_index(index_data)

Fetching mindshare data for @HopiumPapi...
Fetching mindshare data for @JakeGagain...
Fetching mindshare data for @cryptobeastreal...
Fetching mindshare data for @elchartox...
Fetching mindshare data for @issathecooker...
Fetching mindshare data for @realdogen...
Fetching mindshare data for @ProTheDoge...
Fetching mindshare data for @shahh...
Fetching mindshare data for @TeTheGamer...


In [ ]:
detectives = ['@dethective']

index_data = get_index_data(detectives)
df_detectives = create_index(index_data)

Fetching mindshare data for @dethective...


In [ ]:
# Create figure
fig = go.Figure()

# Add trace for detectives
fig.add_trace(
    go.Scatter(
        x=df_detectives.index,
        y=df_detectives['index_mindshare'],
        name='Detectives',
        mode='lines',
        line=dict(
            width=4,  # Increased thickness
            color='#00FF7F',  # Spring green color
        ),
        hovertemplate='Index: %{y:.2f}<br>Date: %{x}<extra></extra>'
    )
)

# Add trace for shillers
fig.add_trace(
    go.Scatter(
        x=df_shillers.index,
        y=df_shillers['index_mindshare'],
        name='Shillers',
        mode='lines',
        line=dict(
            width=4,  # Increased thickness
            color='#FF3333',  # Bright red color
        ),
        hovertemplate='Index: %{y:.2f}<br>Date: %{x}<extra></extra>'
    )
)

# Update layout with transparent background
fig.update_layout(
    title={
        'text': 'Mindshare Index Comparison',
        'font': {'size': 24, 'color': 'white'},
        'y': 0.95
    },
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    yaxis_title='Detectives vs Shillers index',
    xaxis_title='Date',
    height=700,
    width=1200,
    showlegend=True,
    legend=dict(
        font=dict(color='white', size=12),
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(255,255,255,0.2)',
        borderwidth=1
    ),
    hovermode='x unified',
    hoverlabel=dict(bgcolor='rgba(0,0,0,0.8)', font_size=12),
    xaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(255,255,255,0.1)',
        tickfont=dict(color='white'),
        title_font=dict(color='white', size=14),
        showline=False
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(255,255,255,0.1)',
        tickfont=dict(color='white'),
        title_font=dict(color='white', size=14),
        showline=False
    )
)

# Add margins
fig.update_layout(margin=dict(l=60, r=30, t=80, b=60))

# Show the plot
fig.show()